In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1mt8AoUX7lWzNmhhz-U0kK2pUXXXK5NT3", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))


# Sequence Parallelism: Implementing AllGather and Reduce-Scatter

*Notebook 2 of the Sequence Parallelism series — Vizuara GPU Programming Course*

*Estimated time: 45-55 minutes*

---

In the previous notebook, we discovered that LayerNorm and dropout waste memory in tensor parallelism because they require full-size activations on every GPU. We also showed that splitting along the **sequence dimension** gives identical results for these operations.

In this notebook, we implement sequence parallelism from scratch. The key insight is that transitioning between sequence-parallel (SP) layout $(b, s/N, h)$ and tensor-parallel (TP) layout $(b, s, h/N)$ requires two collective operations: **AllGather** and **Reduce-Scatter**. Together, they move exactly the same amount of data as the AllReduce used in standard TP — making sequence parallelism **zero cost** in communication.

In [ ]:
#@title 🎧 Listen: Assistant Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_assistant_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[Open AI Teaching Assistant](https://pods.vizuara.ai/courses/gpu-programming/practice/7/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*

In [ ]:
# Setup: Run this cell first
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import sys

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("No GPU detected. This notebook simulates multi-GPU on CPU.")

print(f"Python {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

torch.manual_seed(42)
np.random.seed(42)

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Data Layouts Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_data_layouts_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. The Two Data Layouts

Sequence parallelism introduces a second way to distribute activations across GPUs:

| Layout | Shape per GPU | Split Dimension | Used For |
|--------|---------------|-----------------|----------|
| Tensor-Parallel (TP) | $(b, s, h/N)$ | Hidden | Attention, MLP |
| Sequence-Parallel (SP) | $(b, s/N, h)$ | Sequence | LayerNorm, Dropout, Residual |

Both layouts distribute $b \times s \times h$ total elements across $N$ GPUs — each GPU holds $b \times s \times h / N$ elements. The difference is **which dimension** is split.

Let us visualize these two layouts.

In [ ]:
#@title 🎧 Code Walkthrough: Data Layouts Config
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_data_layouts_config.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Configuration
B = 1           # batch
S = 16          # sequence length (small for visualization)
H = 8           # hidden dim (small for visualization)
N = 4           # number of simulated GPUs

# Create a full activation tensor
full_tensor = torch.arange(S * H).reshape(1, S, H).float()  # (1, 16, 8)

print(f"Full tensor shape: {full_tensor.shape}")
print(f"Total elements: {full_tensor.numel()}")
print(f"Elements per GPU: {full_tensor.numel() // N}")

In [ ]:
#@title 🎧 What to Look For: Data Layouts Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_data_layouts_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the two layouts

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']  # blue, green, orange, purple

# Full tensor
ax = axes[0]
data = full_tensor[0].numpy()
ax.imshow(data, aspect='auto', cmap='viridis')
ax.set_xlabel('Hidden dimension (h)', fontsize=11)
ax.set_ylabel('Sequence position (s)', fontsize=11)
ax.set_title('Full Activation Tensor\n(b, s, h) = (1, 16, 8)', fontsize=12)

# SP layout: split along sequence
ax = axes[1]
for gpu_id in range(N):
    start = gpu_id * (S // N)
    end = (gpu_id + 1) * (S // N)
    chunk = data[start:end, :]
    # Draw colored rectangle
    rect = plt.Rectangle((0, start), H, S // N, linewidth=2,
                          edgecolor=colors[gpu_id], facecolor=colors[gpu_id], alpha=0.3)
    ax.add_patch(rect)
    ax.text(H / 2, start + (S // N) / 2, f'GPU {gpu_id}\n({B},{S // N},{H})',
            ha='center', va='center', fontsize=9, fontweight='bold',
            color=colors[gpu_id])

ax.imshow(data, aspect='auto', cmap='viridis', alpha=0.5)
ax.set_xlabel('Hidden dimension (h) — FULL', fontsize=11)
ax.set_ylabel('Sequence position (s) — SPLIT', fontsize=11)
ax.set_title('Sequence-Parallel Layout\n(b, s/N, h) per GPU', fontsize=12)

# TP layout: split along hidden
ax = axes[2]
for gpu_id in range(N):
    start = gpu_id * (H // N)
    end = (gpu_id + 1) * (H // N)
    rect = plt.Rectangle((start, 0), H // N, S, linewidth=2,
                          edgecolor=colors[gpu_id], facecolor=colors[gpu_id], alpha=0.3)
    ax.add_patch(rect)
    ax.text(start + (H // N) / 2, S / 2, f'GPU {gpu_id}\n({B},{S},{H // N})',
            ha='center', va='center', fontsize=9, fontweight='bold',
            color=colors[gpu_id], rotation=90)

ax.imshow(data, aspect='auto', cmap='viridis', alpha=0.5)
ax.set_xlabel('Hidden dimension (h) — SPLIT', fontsize=11)
ax.set_ylabel('Sequence position (s) — FULL', fontsize=11)
ax.set_title('Tensor-Parallel Layout\n(b, s, h/N) per GPU', fontsize=12)

plt.suptitle('Two Ways to Distribute the Same Tensor Across 4 GPUs', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Allgather Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_allgather_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Implementing AllGather Along the Sequence Dimension

**AllGather** collects chunks from all GPUs and concatenates them. In sequence parallelism, it transitions from SP layout to full tensor:

$$(b, s/N, h) \xrightarrow{\text{AllGather}} (b, s, h)$$

Each GPU contributes its $s/N$ tokens, and every GPU ends up with the full $s$ tokens.

We simulate this on a single device by explicitly splitting and gathering.

In [ ]:
#@title 🎧 Code Walkthrough: Simulated Tpgroup Allgather
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_simulated_tpgroup_allgather.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class SimulatedTPGroup:
    """Simulates a tensor-parallel group of N GPUs on a single device.
    
    Each 'GPU' is represented by its rank (0 to N-1).
    We store separate tensors for each GPU to simulate distributed memory.
    """
    
    def __init__(self, world_size):
        self.world_size = world_size
    
    def all_gather_along_seq(self, local_tensors):
        """AllGather along the sequence dimension.
        
        Args:
            local_tensors: list of N tensors, each (b, s/N, h)
        
        Returns:
            list of N tensors, each (b, s, h) — every GPU gets the full tensor
        """
        # Concatenate all chunks along sequence dimension
        full_tensor = torch.cat(local_tensors, dim=1)  # (b, s, h)
        # Each GPU gets a copy of the full tensor
        return [full_tensor.clone() for _ in range(self.world_size)]
    
    def reduce_scatter_along_seq(self, partial_tensors):
        """Reduce-Scatter along the sequence dimension.
        
        Args:
            partial_tensors: list of N tensors, each (b, s, h) — partial sums
        
        Returns:
            list of N tensors, each (b, s/N, h) — reduced and scattered
        """
        # Sum all partial tensors
        reduced = sum(partial_tensors)
        # Scatter: split the result along sequence dimension
        seq_len = reduced.shape[1]
        chunk_size = seq_len // self.world_size
        chunks = list(reduced.split(chunk_size, dim=1))
        return chunks
    
    def all_reduce(self, partial_tensors):
        """Standard AllReduce (sum across GPUs, result on all GPUs).
        
        Args:
            partial_tensors: list of N tensors, same shape
        
        Returns:
            list of N tensors, each containing the sum
        """
        reduced = sum(partial_tensors)
        return [reduced.clone() for _ in range(self.world_size)]


# Test
tp_group = SimulatedTPGroup(N)

# Create SP-layout tensors: each GPU has (1, S/N, H)
sp_chunks = [torch.randn(B, S // N, H) for _ in range(N)]
print(f"SP layout: each GPU has shape {sp_chunks[0].shape}")

# AllGather: SP -> full
gathered = tp_group.all_gather_along_seq(sp_chunks)
print(f"After AllGather: each GPU has shape {gathered[0].shape}")

# Verify correctness
expected_full = torch.cat(sp_chunks, dim=1)
for rank in range(N):
    assert torch.allclose(gathered[rank], expected_full)
print(f"AllGather verified: all GPUs have identical full tensors")

In [ ]:
#@title 🎧 Code Walkthrough: Simulated Tpgroup Reduce Scatter
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_simulated_tpgroup_reduce_scatter.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Test Reduce-Scatter

# Create partial sums: each GPU has (1, S, H) — partial results from row-parallel linear
partial_sums = [torch.randn(B, S, H) for _ in range(N)]
print(f"Partial sums: each GPU has shape {partial_sums[0].shape}")

# Reduce-Scatter: sum + split along sequence
scattered = tp_group.reduce_scatter_along_seq(partial_sums)
print(f"After Reduce-Scatter: each GPU has shape {scattered[0].shape}")

# Verify: scattered[rank] should equal sum(partial_sums)[rank's_chunk]
total_sum = sum(partial_sums)
for rank in range(N):
    expected_chunk = total_sum[:, rank * (S // N):(rank + 1) * (S // N), :]
    assert torch.allclose(scattered[rank], expected_chunk)

print(f"Reduce-Scatter verified: each GPU has the correct reduced chunk")

In [ ]:
#@title 🎧 Listen: Key Equivalence Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_key_equivalence_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. The Key Equivalence: AllReduce = Reduce-Scatter + AllGather

Standard tensor parallelism uses an **AllReduce** after the row-parallel linear layer. Sequence parallelism decomposes this into a **Reduce-Scatter** (at the TP exit) and an **AllGather** (at the next TP entry). Let us verify they produce the same final result.

In [ ]:
#@title 🎧 Code Walkthrough: Key Equivalence Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_key_equivalence_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Demonstrate: AllReduce == Reduce-Scatter + AllGather

# Each GPU has a partial sum from the row-parallel linear layer
partial_sums = [torch.randn(B, S, H) for _ in range(N)]

# Method 1: Standard TP — AllReduce
allreduced = tp_group.all_reduce(partial_sums)
# Each GPU now has the full reduced tensor (b, s, h)

# Method 2: Sequence Parallelism — Reduce-Scatter then AllGather
# Step 1: Reduce-Scatter -> each GPU gets (b, s/N, h)
scattered = tp_group.reduce_scatter_along_seq(partial_sums)
# (imagine LayerNorm, dropout, residual happen here on the scattered chunks)
# Step 2: AllGather -> each GPU gets (b, s, h)
regathered = tp_group.all_gather_along_seq(scattered)

# Compare
for rank in range(N):
    max_diff = (allreduced[rank] - regathered[rank]).abs().max().item()
    print(f"GPU {rank}: max difference = {max_diff:.2e}")

print(f"\nAllReduce and Reduce-Scatter + AllGather give identical results!")

In [ ]:
#@title 🎧 Listen: Comm Cost Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_comm_cost_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Communication Cost Analysis

Let us now verify that the communication cost is truly the same.

For a tensor with $M$ elements across $N$ GPUs:

| Operation | Data per GPU |
|-----------|-------------|
| AllReduce | $2 \cdot \frac{N-1}{N} \cdot M$ |
| Reduce-Scatter | $\frac{N-1}{N} \cdot M$ |
| AllGather | $\frac{N-1}{N} \cdot M$ |
| RS + AG total | $2 \cdot \frac{N-1}{N} \cdot M$ |

In [ ]:
#@title 🎧 Code Walkthrough: Comm Cost Calc
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_comm_cost_calc.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def communication_cost(M, N, dtype_bytes=2):
    """Compute communication bytes per GPU for different collective operations."""
    factor = (N - 1) / N
    
    allreduce = 2 * factor * M * dtype_bytes
    reduce_scatter = factor * M * dtype_bytes
    allgather = factor * M * dtype_bytes
    
    return {
        'allreduce': allreduce,
        'reduce_scatter': reduce_scatter,
        'allgather': allgather,
        'rs_plus_ag': reduce_scatter + allgather,
    }

# Our example: (1, 4096, 8192) in BF16
M = 1 * 4096 * 8192
costs = communication_cost(M, 4)

print(f"Tensor: (1, 4096, 8192) = {M:,} elements, N = 4 GPUs, BF16")
print(f"\n{'Operation':<20} {'Data per GPU (MB)':<20}")
print("-" * 42)
for name, value in costs.items():
    print(f"{name:<20} {value / 1e6:<20.2f}")

print(f"\nAllReduce == RS + AG? {abs(costs['allreduce'] - costs['rs_plus_ag']) < 1e-6}")

In [ ]:
#@title 🎧 What to Look For: Comm Cost Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_comm_cost_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the cost equivalence across different N values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

M = 1 * 2048 * 8192  # typical tensor size
ns = [2, 4, 8, 16, 32]

# Left: absolute bytes
ax = axes[0]
ar_costs = [communication_cost(M, n)['allreduce'] / 1e6 for n in ns]
rs_costs = [communication_cost(M, n)['reduce_scatter'] / 1e6 for n in ns]
ag_costs = [communication_cost(M, n)['allgather'] / 1e6 for n in ns]

x = np.arange(len(ns))
width = 0.35

ax.bar(x - width/2, ar_costs, width, label='AllReduce (standard TP)',
       color='#F44336', alpha=0.8)
ax.bar(x + width/2, rs_costs, width, label='Reduce-Scatter',
       color='#2196F3', alpha=0.8)
ax.bar(x + width/2, ag_costs, width, bottom=rs_costs, label='AllGather',
       color='#4CAF50', alpha=0.8)

ax.set_xlabel('TP Degree (N)', fontsize=12)
ax.set_ylabel('Data per GPU (MB)', fontsize=12)
ax.set_title('Communication Cost per GPU', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(ns)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Right: bandwidth utilization factor
ax = axes[1]
factors = [(n - 1) / n for n in ns]
ax.plot(ns, factors, 'o-', color='#1565c0', linewidth=2, markersize=8)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('TP Degree (N)', fontsize=12)
ax.set_ylabel('(N-1)/N factor', fontsize=12)
ax.set_title('Bandwidth Utilization Factor', fontsize=13)
ax.set_ylim(0, 1.1)
ax.grid(alpha=0.3)

for n, f in zip(ns, factors):
    ax.annotate(f'{f:.2f}', (n, f), textcoords="offset points",
                xytext=(0, 10), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("The red bars (AllReduce) exactly equal the stacked blue+green bars (RS + AG).")
print("Sequence parallelism decomposes the AllReduce — no extra communication!")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_todo_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Your Turn: Implement the SP Forward Pass for One Sub-Block

### TODO: Complete the forward pass with sequence parallelism

Implement the forward pass for one sub-block (e.g., the attention sub-block) using sequence parallelism. The flow is:

1. LayerNorm on SP-layout input $(b, s/N, h)$
2. AllGather to get full tensor $(b, s, h)$
3. Column-parallel linear $(b, s, h) \to (b, s, h/N)$
4. Some computation (simulated)
5. Row-parallel linear $(b, s, h/N) \to (b, s, h)$ partial sum
6. Reduce-Scatter back to SP layout $(b, s/N, h)$

In [ ]:
#@title 🎧 Code Walkthrough: Todo Helpers
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_todo_helpers.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class ColumnParallelLinear:
    """Simulates a column-parallel linear layer split across N GPUs.
    
    Full weight: (h, h_out)
    Each GPU stores: (h, h_out/N) — a column slice
    Input: (b, s, h) — full
    Output: (b, s, h_out/N) — split along output dim
    """
    def __init__(self, in_features, out_features, n_gpus):
        self.n_gpus = n_gpus
        self.out_per_gpu = out_features // n_gpus
        # Each GPU's weight shard
        self.weights = [torch.randn(in_features, self.out_per_gpu) * 0.01
                        for _ in range(n_gpus)]
    
    def forward(self, inputs):
        """inputs: list of N tensors, each (b, s, h)"""
        outputs = []
        for rank in range(self.n_gpus):
            out = inputs[rank] @ self.weights[rank]  # (b, s, h_out/N)
            outputs.append(out)
        return outputs


class RowParallelLinear:
    """Simulates a row-parallel linear layer.
    
    Full weight: (h_in, h)
    Each GPU stores: (h_in/N, h) — a row slice
    Input: (b, s, h_in/N) — split
    Output: (b, s, h) — partial sum (needs reduce)
    """
    def __init__(self, in_features, out_features, n_gpus):
        self.n_gpus = n_gpus
        self.in_per_gpu = in_features // n_gpus
        self.weights = [torch.randn(self.in_per_gpu, out_features) * 0.01
                        for _ in range(n_gpus)]
    
    def forward(self, inputs):
        """inputs: list of N tensors, each (b, s, h_in/N)
        Returns: list of N tensors, each (b, s, h) — partial sums"""
        outputs = []
        for rank in range(self.n_gpus):
            out = inputs[rank] @ self.weights[rank]  # (b, s, h)
            outputs.append(out)
        return outputs


def sp_forward_subblock(sp_inputs, ln, col_linear, row_linear, tp_group):
    """Forward pass for one sub-block with Tensor + Sequence Parallelism.
    
    Args:
        sp_inputs: list of N tensors, each (b, s/N, h) — SP layout
        ln: nn.LayerNorm module
        col_linear: ColumnParallelLinear
        row_linear: RowParallelLinear
        tp_group: SimulatedTPGroup
    
    Returns:
        list of N tensors, each (b, s/N, h) — SP layout output
    """
    N = tp_group.world_size
    
    # ============ TODO ============
    # Step 1: Apply LayerNorm locally on each GPU's SP chunk
    #         Each GPU has (b, s/N, h) — apply ln() to each
    ln_outputs = ???  # YOUR CODE: list of N tensors, each (b, s/N, h)
    
    # Step 2: AllGather along sequence dimension
    #         (b, s/N, h) per GPU -> (b, s, h) per GPU
    full_tensors = ???  # YOUR CODE: use tp_group.all_gather_along_seq
    
    # Step 3: Column-parallel linear
    #         (b, s, h) per GPU -> (b, s, h_out/N) per GPU
    col_outputs = ???  # YOUR CODE: use col_linear.forward
    
    # Step 4: Simulated computation (e.g., activation function)
    activated = [F.gelu(out) for out in col_outputs]
    
    # Step 5: Row-parallel linear
    #         (b, s, h_out/N) per GPU -> (b, s, h) per GPU (partial sums)
    partial_sums = ???  # YOUR CODE: use row_linear.forward
    
    # Step 6: Reduce-Scatter along sequence dimension
    #         (b, s, h) partial sums -> (b, s/N, h) per GPU (reduced)
    sp_outputs = ???  # YOUR CODE: use tp_group.reduce_scatter_along_seq
    # ==============================
    
    return sp_outputs


# Test it
ln = nn.LayerNorm(H)
col_lin = ColumnParallelLinear(H, H, N)
row_lin = RowParallelLinear(H, H, N)

# SP inputs: each GPU has (1, S/N, H)
sp_in = [torch.randn(B, S // N, H) for _ in range(N)]

try:
    sp_out = sp_forward_subblock(sp_in, ln, col_lin, row_lin, tp_group)
    print(f"Input shape per GPU:  {sp_in[0].shape}")
    print(f"Output shape per GPU: {sp_out[0].shape}")
    assert sp_out[0].shape == (B, S // N, H)
    print("SP forward pass verified!")
except Exception as e:
    print(f"Error: {e}")
    print("Complete the TODO above.")

In [ ]:
#@title 🎧 Listen: Todo Post
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_todo_post.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


---
### Stop and Think

1. In step 2 (AllGather), why does every GPU need the full sequence? What would break if we skipped it?
2. In step 6 (Reduce-Scatter), why is the sum needed? Where do the partial sums come from?
3. Could we do AllGather and Reduce-Scatter in the opposite order? What would that mean?

---

In [ ]:
#@title 🎧 Code Walkthrough: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_todo_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
def sp_forward_subblock(sp_inputs, ln, col_linear, row_linear, tp_group):
    """Forward pass for one sub-block with Tensor + Sequence Parallelism."""
    N = tp_group.world_size
    
    # Step 1: LayerNorm locally on each GPU's SP chunk
    ln_outputs = [ln(sp_inputs[rank]) for rank in range(N)]
    
    # Step 2: AllGather along sequence dimension
    full_tensors = tp_group.all_gather_along_seq(ln_outputs)
    
    # Step 3: Column-parallel linear
    col_outputs = col_linear.forward(full_tensors)
    
    # Step 4: Simulated computation (activation function)
    activated = [F.gelu(out) for out in col_outputs]
    
    # Step 5: Row-parallel linear (produces partial sums)
    partial_sums = row_linear.forward(activated)
    
    # Step 6: Reduce-Scatter along sequence dimension
    sp_outputs = tp_group.reduce_scatter_along_seq(partial_sums)
    
    return sp_outputs


# Test
sp_out = sp_forward_subblock(sp_in, ln, col_lin, row_lin, tp_group)
print(f"Input shape per GPU:  {sp_in[0].shape}")
print(f"Output shape per GPU: {sp_out[0].shape}")
assert sp_out[0].shape == (B, S // N, H)
print(f"SP forward pass verified!")

In [ ]:
#@title 🎧 Narration: Memory Tracking Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_memory_tracking_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Comparing Standard TP vs TP+SP: Memory Tracking

Let us now run both approaches side by side and track the peak memory stored on each GPU at every stage of the forward pass.

In [ ]:
#@title 🎧 Code Walkthrough: Memory Tracking Funcs
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_memory_tracking_funcs.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def trace_memory_standard_tp(b, s, h, N):
    """Trace memory stored on GPU 0 through a standard TP forward pass."""
    stages = []
    elements = lambda shape: np.prod(shape)
    
    # Input: full tensor on each GPU (after AllReduce from previous layer)
    stages.append(('Input (full)', elements((b, s, h)), 'non-tp'))
    
    # LayerNorm: stores input for backward — full tensor
    stages.append(('LayerNorm input', elements((b, s, h)), 'non-tp'))
    
    # Column-parallel linear: output split
    stages.append(('Col-linear output', elements((b, s, h // N)), 'tp'))
    
    # Activation function: stored for backward
    stages.append(('GeLU input', elements((b, s, h // N)), 'tp'))
    
    # Row-parallel linear: partial sum (full size)
    stages.append(('Row-linear output', elements((b, s, h)), 'non-tp'))
    
    # AllReduce: no extra storage (in-place)
    stages.append(('After AllReduce', elements((b, s, h)), 'non-tp'))
    
    # Dropout: full tensor + mask
    stages.append(('Dropout input', elements((b, s, h)), 'non-tp'))
    stages.append(('Dropout mask', elements((b, s, h)), 'non-tp'))
    
    return stages


def trace_memory_tp_sp(b, s, h, N):
    """Trace memory stored on GPU 0 through a TP+SP forward pass."""
    stages = []
    elements = lambda shape: np.prod(shape)
    
    # Input: SP layout
    stages.append(('Input (SP)', elements((b, s // N, h)), 'sp'))
    
    # LayerNorm: SP layout — stores input for backward
    stages.append(('LayerNorm input', elements((b, s // N, h)), 'sp'))
    
    # AllGather: temporarily full
    stages.append(('After AllGather', elements((b, s, h)), 'comm'))
    
    # Column-parallel linear: output split
    stages.append(('Col-linear output', elements((b, s, h // N)), 'tp'))
    
    # Activation function
    stages.append(('GeLU input', elements((b, s, h // N)), 'tp'))
    
    # Row-parallel linear: partial sum
    stages.append(('Row-linear partial', elements((b, s, h)), 'comm'))
    
    # Reduce-Scatter: back to SP layout
    stages.append(('After Reduce-Scatter', elements((b, s // N, h)), 'sp'))
    
    # Dropout: SP layout + mask
    stages.append(('Dropout input', elements((b, s // N, h)), 'sp'))
    stages.append(('Dropout mask', elements((b, s // N, h)), 'sp'))
    
    return stages


# Compare at realistic scale
b, s, h, N = 1, 2048, 4096, 4

tp_stages = trace_memory_standard_tp(b, s, h, N)
sp_stages = trace_memory_tp_sp(b, s, h, N)

print(f"{'Standard TP':<40} {'TP + SP':<40}")
print(f"{'Stage':<25} {'Elements':>12}    {'Stage':<25} {'Elements':>12}")
print("-" * 85)

max_rows = max(len(tp_stages), len(sp_stages))
for i in range(max_rows):
    if i < len(tp_stages):
        name1, elem1, _ = tp_stages[i]
        left = f"{name1:<25} {elem1:>12,}"
    else:
        left = " " * 40
    
    if i < len(sp_stages):
        name2, elem2, _ = sp_stages[i]
        right = f"{name2:<25} {elem2:>12,}"
    else:
        right = ""
    
    print(f"{left}    {right}")

tp_total = sum(e for _, e, cat in tp_stages if cat == 'non-tp')
sp_total = sum(e for _, e, cat in sp_stages if cat == 'sp')

print(f"\nTotal non-TP stored elements (standard TP): {tp_total:,}")
print(f"Total SP stored elements (TP + SP):          {sp_total:,}")
print(f"Reduction: {tp_total / sp_total:.1f}x")

In [ ]:
#@title 🎧 Narration: Viz Sp Forward Pass Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_viz_sp_forward_pass_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Visualizing the SP Forward Pass

Let us create a visual trace showing how data flows through a sub-block with sequence parallelism.

In [ ]:
#@title 🎧 What to Look For: Viz Sp Forward Pass Plot
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_viz_sp_forward_pass_plot.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the data flow in a TP+SP sub-block

fig, ax = plt.subplots(1, 1, figsize=(16, 8))

# Define stages and shapes
stages = [
    ('SP Input\n(b, s/N, h)', 'green', 0.25),
    ('LayerNorm\n(local)', 'green', 0.25),
    ('AllGather', 'gray', 0.5),
    ('Full Tensor\n(b, s, h)', 'orange', 1.0),
    ('Col-Parallel\nLinear', 'blue', 0.25),
    ('TP Output\n(b, s, h/N)', 'blue', 0.25),
    ('GeLU', 'blue', 0.25),
    ('Row-Parallel\nLinear', 'blue', 0.25),
    ('Partial Sum\n(b, s, h)', 'orange', 1.0),
    ('Reduce-\nScatter', 'gray', 0.5),
    ('SP Output\n(b, s/N, h)', 'green', 0.25),
]

color_map = {
    'green': '#4CAF50',
    'blue': '#2196F3',
    'orange': '#FF9800',
    'gray': '#9E9E9E',
}

y_positions = np.linspace(7, 0, len(stages))

for i, (label, color, width_factor) in enumerate(stages):
    y = y_positions[i]
    w = 3 * width_factor
    x_center = 5
    
    rect = mpatches.FancyBboxPatch(
        (x_center - w/2, y - 0.2), w, 0.4,
        boxstyle="round,pad=0.05",
        facecolor=color_map[color], alpha=0.7,
        edgecolor='black', linewidth=1.5
    )
    ax.add_patch(rect)
    ax.text(x_center, y, label, ha='center', va='center',
            fontsize=9, fontweight='bold')
    
    # Arrow to next stage
    if i < len(stages) - 1:
        ax.annotate('', xy=(x_center, y_positions[i+1] + 0.25),
                    xytext=(x_center, y - 0.25),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    
    # Memory label on the right
    if color == 'green':
        mem_label = 'Memory: b*s/N*h'
        mem_color = '#4CAF50'
    elif color == 'blue':
        mem_label = 'Memory: b*s*h/N'
        mem_color = '#2196F3'
    elif color == 'orange':
        mem_label = 'Memory: b*s*h (temp)'
        mem_color = '#FF9800'
    else:
        mem_label = 'Communication'
        mem_color = '#9E9E9E'
    
    ax.text(8, y, mem_label, fontsize=9, color=mem_color, fontweight='bold')

# Regime labels on the left
ax.text(1.0, 6.8, 'SP Regime', fontsize=12, fontweight='bold', color='#4CAF50',
        rotation=90, va='center')
ax.text(1.0, 3.5, 'TP Regime', fontsize=12, fontweight='bold', color='#2196F3',
        rotation=90, va='center')
ax.text(1.0, 0.5, 'SP Regime', fontsize=12, fontweight='bold', color='#4CAF50',
        rotation=90, va='center')

ax.set_xlim(0, 12)
ax.set_ylim(-0.8, 8)
ax.set_title('Data Flow Through a TP+SP Sub-Block (One GPU\'s View)', fontsize=14)
ax.axis('off')

# Legend
legend_items = [
    mpatches.Patch(color='#4CAF50', alpha=0.7, label='Sequence-Parallel (b, s/N, h)'),
    mpatches.Patch(color='#2196F3', alpha=0.7, label='Tensor-Parallel (b, s, h/N)'),
    mpatches.Patch(color='#FF9800', alpha=0.7, label='Full tensor (temporary)'),
    mpatches.Patch(color='#9E9E9E', alpha=0.7, label='Communication'),
]
ax.legend(handles=legend_items, loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Narration: Full Sp Forward Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_full_sp_forward_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Running the Full Simulated SP Forward Pass

Let us put everything together and run a full forward pass through two sub-blocks (attention + MLP) with sequence parallelism, tracking shapes at every step.

In [ ]:
#@title 🎧 Code Walkthrough: Full Sp Forward Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_22_full_sp_forward_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Full TP+SP forward pass through both sub-blocks

h_demo = 64   # small for demo
s_demo = 32
b_demo = 1
n_demo = 4

tp_group_demo = SimulatedTPGroup(n_demo)

# LayerNorms
ln1 = nn.LayerNorm(h_demo)
ln2 = nn.LayerNorm(h_demo)

# Attention sub-block (simplified as two linears)
attn_col = ColumnParallelLinear(h_demo, h_demo, n_demo)
attn_row = RowParallelLinear(h_demo, h_demo, n_demo)

# MLP sub-block
mlp_col = ColumnParallelLinear(h_demo, h_demo * 4, n_demo)
mlp_row = RowParallelLinear(h_demo * 4, h_demo, n_demo)

# Initial SP input: each GPU has (b, s/N, h)
sp_input = [torch.randn(b_demo, s_demo // n_demo, h_demo) for _ in range(n_demo)]

print(f"=== Full TP+SP Transformer Layer ===")
print(f"Config: b={b_demo}, s={s_demo}, h={h_demo}, N={n_demo}")
print(f"")

# --- Attention sub-block ---
print(f"Input (SP):              {[t.shape for t in sp_input]}")
residual = [t.clone() for t in sp_input]

# LayerNorm 1
ln1_out = [ln1(sp_input[r]) for r in range(n_demo)]
print(f"After LayerNorm 1 (SP):  {[t.shape for t in ln1_out]}")

# AllGather
gathered1 = tp_group_demo.all_gather_along_seq(ln1_out)
print(f"After AllGather (full):  {[t.shape for t in gathered1]}")

# Column-parallel attention
attn_tp = attn_col.forward(gathered1)
print(f"After Col-Parallel (TP): {[t.shape for t in attn_tp]}")

# GeLU (simulating attention computation)
attn_act = [F.gelu(t) for t in attn_tp]

# Row-parallel + Reduce-Scatter
attn_partial = attn_row.forward(attn_act)
print(f"After Row-Parallel:      {[t.shape for t in attn_partial]}  (partial sums)")

attn_sp = tp_group_demo.reduce_scatter_along_seq(attn_partial)
print(f"After Reduce-Scatter:    {[t.shape for t in attn_sp]}")

# Dropout + Residual (SP)
attn_out = [F.dropout(attn_sp[r], p=0.1, training=False) + residual[r] for r in range(n_demo)]
print(f"After Dropout+Residual:  {[t.shape for t in attn_out]}")

print(f"")

# --- MLP sub-block ---
residual2 = [t.clone() for t in attn_out]

ln2_out = [ln2(attn_out[r]) for r in range(n_demo)]
print(f"After LayerNorm 2 (SP):  {[t.shape for t in ln2_out]}")

gathered2 = tp_group_demo.all_gather_along_seq(ln2_out)
print(f"After AllGather (full):  {[t.shape for t in gathered2]}")

mlp_tp = mlp_col.forward(gathered2)
print(f"After MLP Col (TP):      {[t.shape for t in mlp_tp]}")

mlp_act = [F.gelu(t) for t in mlp_tp]

mlp_partial = mlp_row.forward(mlp_act)
print(f"After MLP Row:           {[t.shape for t in mlp_partial]}  (partial sums)")

mlp_sp = tp_group_demo.reduce_scatter_along_seq(mlp_partial)
print(f"After Reduce-Scatter:    {[t.shape for t in mlp_sp]}")

mlp_out = [F.dropout(mlp_sp[r], p=0.1, training=False) + residual2[r] for r in range(n_demo)]
print(f"After Dropout+Residual:  {[t.shape for t in mlp_out]}")

print(f"\nOutput (SP):             {[t.shape for t in mlp_out]}")
print(f"\nAll SP operations use shape (1, {s_demo // n_demo}, {h_demo}) = {n_demo}x smaller!")

In [ ]:
#@title 🎧 Narration: Memory Savings Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_23_memory_savings_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Measuring the Memory Savings

Let us compute the total activation memory savings from using TP+SP vs standard TP for a realistic model configuration.

In [ ]:
#@title 🎧 Code Walkthrough: Memory Savings Calc
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_24_memory_savings_calc.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def compare_tp_vs_sp(hidden_dim, seq_len, batch_size, num_layers, tp_degree, dtype_bytes=2):
    """Compare activation memory between standard TP and TP+SP."""
    b, s, h, N, L = batch_size, seq_len, hidden_dim, tp_degree, num_layers
    
    # Per layer, standard TP:
    # Non-TP stored: 2 LN inputs + 2 dropout inputs + 1 residual = 5 * (b*s*h)
    # Plus 2 dropout masks = 2 * (b*s*h) at 1 byte
    # TP stored: attn_qkv + attn_scores + attn_out + mlp_up + mlp_gelu
    #   = 3*(b*s*h/N) + (b*heads_per_gpu*s*s) + (b*s*h/N) + 2*(b*s*4h/N)
    # We simplify: just count non-TP vs SP
    
    non_tp_per_layer_std = 5 * b * s * h * dtype_bytes + 2 * b * s * h  # masks at 1 byte
    non_tp_per_layer_sp = 5 * b * (s // N) * h * dtype_bytes + 2 * b * (s // N) * h
    
    total_std = non_tp_per_layer_std * L
    total_sp = non_tp_per_layer_sp * L
    saved = total_std - total_sp
    
    return {
        'std_tp_gb': total_std / 1e9,
        'tp_sp_gb': total_sp / 1e9,
        'saved_gb': saved / 1e9,
        'reduction_factor': total_std / total_sp if total_sp > 0 else float('inf'),
    }


# Compare across model sizes
configs = [
    ('7B',   4096,  32),
    ('30B',  7168,  60),
    ('70B',  8192,  80),
    ('175B', 12288, 96),
]

N = 8
print(f"TP degree N = {N}, sequence length = 2048, batch = 1, BF16")
print(f"")
print(f"{'Model':<8} {'Standard TP (GB)':<18} {'TP + SP (GB)':<15} {'Saved (GB)':<12} {'Reduction':<10}")
print("-" * 65)

for name, h, L in configs:
    r = compare_tp_vs_sp(h, 2048, 1, L, N)
    print(f"{name:<8} {r['std_tp_gb']:<18.2f} {r['tp_sp_gb']:<15.2f} {r['saved_gb']:<12.2f} {r['reduction_factor']:<10.1f}x")

In [ ]:
#@title 🎧 What to Look For: Memory Savings Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_25_memory_savings_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Final visualization: memory comparison

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

model_names = [c[0] for c in configs]
std_tp_vals = [compare_tp_vs_sp(c[1], 2048, 1, c[2], 8)['std_tp_gb'] for c in configs]
tp_sp_vals = [compare_tp_vs_sp(c[1], 2048, 1, c[2], 8)['tp_sp_gb'] for c in configs]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, std_tp_vals, width, label='Standard TP (replicated)',
               color='#F44336', alpha=0.85)
bars2 = ax.bar(x + width/2, tp_sp_vals, width, label='TP + SP (distributed)',
               color='#4CAF50', alpha=0.85)

# Add value labels
for bar, val in zip(bars1, std_tp_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')
for bar, val in zip(bars2, tp_sp_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Model Size', fontsize=12)
ax.set_ylabel('Non-TP Activation Memory per GPU (GB)', fontsize=12)
ax.set_title('Activation Memory: Standard TP vs TP + Sequence Parallelism\n(N=8, s=2048, b=1, BF16)',
             fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"For the 175B model with TP=8:")
print(f"  Standard TP wastes {std_tp_vals[-1]:.1f} GB per GPU on replicated activations")
print(f"  TP + SP reduces this to {tp_sp_vals[-1]:.1f} GB — an {std_tp_vals[-1] / tp_sp_vals[-1]:.0f}x reduction")
print(f"  Savings: {std_tp_vals[-1] - tp_sp_vals[-1]:.1f} GB freed per GPU")

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_26_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 10. Summary

In this notebook, we implemented sequence parallelism from scratch:

**What we built:**
- **AllGather along sequence**: $(b, s/N, h) \to (b, s, h)$ — collects the full sequence before entering the TP region
- **Reduce-Scatter along sequence**: $(b, s, h) \to (b, s/N, h)$ — reduces partial sums and distributes along sequence
- **Full TP+SP sub-block**: LayerNorm (SP) -> AllGather -> Column-linear (TP) -> Row-linear (TP) -> Reduce-Scatter -> Dropout+Residual (SP)

**Key results:**
- AllReduce = Reduce-Scatter + AllGather in total bandwidth (zero extra cost)
- Non-TP activation memory reduced by factor of $N$
- For a 175B model with TP=8: saves approximately 20 GB per GPU

**In the next notebook**, we will build a complete transformer layer combining TP and SP, run the full forward pass, and measure end-to-end memory savings with actual GPU profiling.